# 02.03 — Manual review và khóa Core

Câu hỏi: ngoài metric, 50 câu IT đã khóa cho thấy lỗi dịch và lỗi kỹ thuật gì? Reviewer điền CSV bên ngoài notebook; notebook chỉ kiểm tra giá trị hợp lệ và đọc decision đã tạo thủ công.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'configs/phase02_protocol.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository root.')
reviews = [p for p in sorted((ROOT / 'reviews').glob('phase02_*_review.csv')) if p.name != 'phase02_it_validation_review_template.csv']
for path in reviews:
    frame = pd.read_csv(path, keep_default_na=False)
    required = {'translation_error', 'technical_error'}
    if len(frame) != 50 or not required.issubset(frame.columns) or not set(frame.translation_error).issubset({'yes','no','unclear'}) or not set(frame.technical_error).issubset({'yes','no','unclear'}):
        print('INCOMPLETE OR INVALID:', path.relative_to(ROOT))
        continue
    print(path.relative_to(ROOT), '| translation:', frame.translation_error.value_counts().to_dict(), '| technical:', frame.technical_error.value_counts().to_dict())

reviews\phase02_envit5_en_to_vi_final-epoch-1_review.csv | translation: {'no': 32, 'yes': 18} | technical: {'no': 36, 'yes': 12, 'unclear': 2}
reviews\phase02_envit5_vi_to_en_final-epoch-1_review.csv | translation: {'no': 35, 'yes': 14, 'unclear': 1} | technical: {'no': 44, 'yes': 6}


In [2]:
import json
decision_path = ROOT / 'evidence/phase02/core_mt_decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text(encoding='utf-8'))
    assert decision.get('status') == 'frozen_before_final_evaluation'
    decision['selections']
else:
    print('No Core decision yet. Do not open final test.')

No Core decision yet. Do not open final test.


Viết quyết định sau khi đối chiếu adapted validation, General Validation, review và provenance. Decision phải chọn một Core/checkpoint cho từng chiều rồi chạy gate final; không có cell nào trong notebook mở final test.